# Day 04 · Cleaning and quarantine

**The idea:** every raw line ends up in **exactly one** place:
- **clean**: it passed every rule, or
- **quarantine**: it broke at least one rule. It's kept, together with the **reasons**, so someone can fix it later.

Nothing is silently thrown away.

| Task | Concept |
|---|---|
| 1 | Clean text: `trim`, `lower`, `initcap`, and keep the original value |
| 2 | Missing values: null vs blank, and `coalesce` |
| 3 | Text that failed to become a number |
| 4 | Business rules as true/false flags, and the null trap |
| 5 | Unknown products and orders with an **anti join** |
| 6 | One array of rejection reasons per line, then split into clean and quarantine |
| 7 | Write Parquet, build a rule-count report, and check the totals add up |

Run on the **pyspark-course** cluster, not Serverless.

In [0]:
from pyspark.sql import functions as F
from decimal import Decimal

RAW = "/Volumes/dbw_pyspark_course/retail/raw"

# Today's outputs go into the scratch volume (created on Day 02)
spark.sql("CREATE VOLUME IF NOT EXISTS dbw_pyspark_course.retail.scratch")
OUT = "/Volumes/dbw_pyspark_course/retail/scratch/day04"

### Task 1 · Clean text values

**Problem:** In `customers.csv`, the cities are written inconsistently (`" Delhi "`, `mumbai`, `CHENNAI`), and the emails have extra spaces. Clean them:
- `trim`: remove spaces at the start and end
- `lower`: make all letters lowercase (for emails)
- `initcap`: capitalise the first letter of each word (for cities)

Don't touch `customer_id`, and keep the original values in `email_raw` and `city_raw`.

**Expected:**
- Searching for `city == "Mumbai"` finds **0** customers before cleaning (they're stored as `mumbai`) and **20** after.
- After cleaning there are 4 cities with 20 customers each: Bengaluru, Chennai, Delhi, Mumbai.

In [0]:
customers = (spark.read
    .option("header", True)
    .schema("customer_id STRING, name STRING, email STRING, city STRING, signup_date DATE, updated_at TIMESTAMP")
    .csv(f"{RAW}/customers.csv"))

# Wrap the values in [ ] so the extra spaces are visible
customers.select(
    "customer_id",
    F.concat(F.lit("["), F.col("email"), F.lit("]")).alias("email"),
    F.concat(F.lit("["), F.col("city"), F.lit("]")).alias("city"),
).show(5, truncate=False)

In [0]:
customers_clean = (customers
    .withColumn("email_raw", F.col("email"))                   # keep the original value
    .withColumn("city_raw",  F.col("city"))
    .withColumn("email", F.lower(F.trim(F.col("email"))))      # remove spaces, then lowercase
    .withColumn("city",  F.initcap(F.trim(F.col("city")))))    # remove spaces, then Capitalise Each Word

customers_clean.select("customer_id", "email_raw", "email", "city_raw", "city").show(5, truncate=False)

In [0]:
# Why it matters: a simple search only works on the clean data
print("'Mumbai' before cleaning:", customers.filter(F.col("city") == "Mumbai").count())
print("'Mumbai' after cleaning :", customers_clean.filter(F.col("city") == "Mumbai").count())

customers_clean.groupBy("city").count().show()

### Task 2 · Missing values: null vs blank, and `coalesce`

**Problem:**
1. A `line_id` can be missing in two ways: **null** (no value at all) or **blank** (`""` or only spaces). Flag both.
2. `coalesce(a, b, c)` returns the **first value that is not null**, checked from left to right. Use it to pick the best contact for a customer.
3. Find the customers with no email. Should they be rejected?

**Expected:**
1. `L1` isn't missing. The null, the `""` and the `"   "` values are all **missing**. A null fails `== ""`, so you need **both** checks.
2. `best_contact`: C1 → its email, C2 → its phone, C3 → `no contact`.
3. 4 customers (C017, C034, C051, C068) have no email. That is **allowed**, because a customer without an email can still buy. A line **without a `line_id` is rejected**, because we can't identify it or trace it back.

In [0]:
ids = spark.createDataFrame([("L1",), (None,), ("",), ("   ",)], "line_id STRING")

ids = ids.withColumn("is_null",  F.col("line_id").isNull())
ids = ids.withColumn("is_blank", F.trim(F.col("line_id")) == "")
ids = ids.withColumn("missing",  F.col("line_id").isNull() | (F.trim(F.col("line_id")) == ""))

ids.show()

In [0]:
contacts = spark.createDataFrame([
    ("C1", "a@example.test", "111"),
    ("C2", None,             "222"),
    ("C3", None,             None),
], "customer_id STRING, email STRING, phone STRING")

# Take email if present, otherwise phone, otherwise the text "no contact"
contacts.withColumn("best_contact", F.coalesce(F.col("email"), F.col("phone"), F.lit("no contact"))).show()

In [0]:
# Real customers with no email: allowed, they are NOT rejected
customers_clean.filter(F.col("email").isNull()).select("customer_id", "name", "email").show()

### Task 3 · Text that failed to become a number

**Problem:** After `cast("int")`, a null quantity can mean two different things:
- the original text was **missing**, or
- the text was **not a number** (`two`), so the cast failed.

Tell them apart.

**Expected:**
- `L1` → 2, OK.
- `L2` → null with **`cast_failed = true`**. This is a *parse problem*: the data arrived broken.
- `L3` → null with `cast_failed = false`. The value was simply missing.

The rule: the original text is **not null**, but the result **is null** → the cast failed.

In [0]:
raw_qty = spark.createDataFrame(
    [("L1", "2"), ("L2", "two"), ("L3", None)],
    "line_id STRING, quantity_text STRING")

raw_qty = raw_qty.withColumn("quantity", F.col("quantity_text").cast("int"))
raw_qty = raw_qty.withColumn("cast_failed", F.col("quantity_text").isNotNull() & F.col("quantity").isNull())

raw_qty.show()

### Load the order data (used in Tasks 4–7)

We read `order_items` with an explicit schema (Day 02) and add `source_file`, so every quarantined line can be traced back to the file it came from. `products` and `orders` are needed to check that the IDs exist.

**Expected:** 785 order lines · 20 products · 300 orders.

In [0]:
order_items = (spark.read
    .option("header", True)
    .schema("line_id STRING, order_id STRING, product_id STRING, quantity INT, unit_price DECIMAL(12,2), discount_pct DECIMAL(5,2)")
    .csv(f"{RAW}/order_items.csv")
    .withColumn("source_file", F.col("_metadata.file_path")))   # where each line came from

products = (spark.read
    .option("header", True)
    .schema("product_id STRING, product_name STRING, category STRING, list_price DECIMAL(12,2), cost_price DECIMAL(12,2)")
    .csv(f"{RAW}/products.csv"))

orders = (spark.read
    .option("header", True)
    .schema("order_id STRING, customer_id STRING, store_id STRING, order_ts TIMESTAMP, status STRING, channel STRING")
    .csv(f"{RAW}/orders.csv"))

print("order lines:", order_items.count())
print("products   :", products.count())
print("orders     :", orders.count())

### Task 4 · Business rules as true/false flags

**Problem:** Turn each rule into its own true/false column:

| Flag | A line is bad when… |
|---|---|
| `missing_line_id`, `missing_order_id`, `missing_product_id` | the ID is null or blank |
| `bad_quantity` | quantity is missing or ≤ 0 |
| `bad_price` | unit_price is missing or < 0 |
| `bad_discount` | discount is missing, < 0 or > 1 |

**The null trap:** first, see why every rule needs an explicit `isNull()` check.

**Expected:**
- Test data: **without** `isNull()`, the null quantity (`T3`) gets `bad_quantity = null` rather than `true`. No reason would be recorded, so the line would **sneak into the clean data**. With `isNull()`, it's flagged.
- Real data: **8** lines fail these rules: `BAD00`–`BAD02` (quantity −1), `BAD03`–`BAD05` (discount 1.50) and `BAD08`–`BAD09` (missing order_id).

In [0]:
test = spark.createDataFrame([("T1", 2), ("T2", 0), ("T3", None)], "line_id STRING, quantity INT")

# WITHOUT a null check: T3 shows null, not true
test.withColumn("bad_quantity", F.col("quantity") <= 0).show()

# WITH a null check: T3 is correctly flagged
test.withColumn("bad_quantity", F.col("quantity").isNull() | (F.col("quantity") <= 0)).show()

In [0]:
# One flag column per rule
flagged = (order_items
    .withColumn("missing_line_id",    F.col("line_id").isNull()    | (F.trim("line_id") == ""))
    .withColumn("missing_order_id",   F.col("order_id").isNull()   | (F.trim("order_id") == ""))
    .withColumn("missing_product_id", F.col("product_id").isNull() | (F.trim("product_id") == ""))
    .withColumn("bad_quantity",       F.col("quantity").isNull()   | (F.col("quantity") <= 0))
    .withColumn("bad_price",          F.col("unit_price").isNull() | (F.col("unit_price") < 0))
    .withColumn("bad_discount",       F.col("discount_pct").isNull() | (F.col("discount_pct") < 0) | (F.col("discount_pct") > 1)))

# Lines that break at least one of these rules
flagged.filter(
    F.col("missing_line_id") | F.col("missing_order_id") | F.col("missing_product_id") |
    F.col("bad_quantity") | F.col("bad_price") | F.col("bad_discount")
).select("line_id", "order_id", "quantity", "discount_pct", "bad_quantity", "bad_discount", "missing_order_id").show()

### Task 5 · Unknown products and orders with an anti join

**Problem:** A line can have valid values but point to a product or order that **doesn't exist**. This is a *referential-integrity* problem.

A **left anti join** returns the rows on the left that have **no match** on the right. That's exactly the "not found" rows.

**Expected:**
- Unknown products: **`BAD06`** and **`BAD07`**, both with `P999`.
- Unknown orders: **`BAD08`** and **`BAD09`**, whose `order_id` is null. **A null never matches anything in a join.** These two are already flagged as `missing_order_id`, so in the flag below we count `unknown_order` only when the ID is present.

In [0]:
# Lines whose product_id is not in the products table
unknown_products = order_items.join(products, on="product_id", how="left_anti")
unknown_products.select("line_id", "product_id").show()

In [0]:
# Lines whose order_id is not in the orders table
unknown_orders = order_items.join(orders, on="order_id", how="left_anti")
unknown_orders.select("line_id", "order_id").show()

In [0]:
# To get flag columns, LEFT join a list of known IDs and see which lines found no match.
# products and orders have one row per ID, so this join does not add extra rows.
known_products = products.select("product_id").withColumn("product_found", F.lit(True))
known_orders   = orders.select("order_id").withColumn("order_found", F.lit(True))

checked = (flagged
    .join(known_products, on="product_id", how="left")
    .join(known_orders,   on="order_id",   how="left")
    .withColumn("unknown_product", F.col("product_found").isNull() & ~F.col("missing_product_id"))
    .withColumn("unknown_order",   F.col("order_found").isNull()   & ~F.col("missing_order_id")))

checked.filter(F.col("unknown_product") | F.col("unknown_order")).select("line_id", "product_id", "unknown_product").show()

### Task 6 · Collect every reason, then split into clean and quarantine

**Problem:** Put all the reasons that apply to a line into one **array** column, `reasons`:
- `F.when(flag, "reason")` gives the reason text when the flag is true, and **null** when it isn't.
- `F.array(...)` puts them all in a list.
- `F.array_compact(...)` removes the nulls.

A line with an **empty** array is clean; anything else goes to quarantine.

**Expected:**
- **775 clean** and **10 quarantine** lines. Each quarantined line shows its reason(s) and its `source_file`.
- A test line that breaks **two** rules keeps **both** reasons: `[bad_quantity, bad_price]`. That's why adding up the per-rule counts can give **more** than the number of quarantined rows.

In [0]:
labelled = checked.withColumn("reasons", F.array_compact(F.array(
    F.when(F.col("missing_line_id"),    F.lit("missing_line_id")),
    F.when(F.col("missing_order_id"),   F.lit("missing_order_id")),
    F.when(F.col("missing_product_id"), F.lit("missing_product_id")),
    F.when(F.col("bad_quantity"),       F.lit("bad_quantity")),
    F.when(F.col("bad_price"),          F.lit("bad_price")),
    F.when(F.col("bad_discount"),       F.lit("bad_discount")),
    F.when(F.col("unknown_product"),    F.lit("unknown_product")),
    F.when(F.col("unknown_order"),      F.lit("unknown_order")),
)))

accepted   = labelled.filter(F.size("reasons") == 0)   # no reasons -> clean
quarantine = labelled.filter(F.size("reasons") > 0)    # at least one reason -> quarantine

print("accepted  :", accepted.count())
print("quarantine:", quarantine.count())

In [0]:
quarantine.select("line_id", "order_id", "product_id", "quantity", "discount_pct", "reasons", "source_file").show(truncate=False)

In [0]:
# A line that breaks TWO rules keeps both reasons
two_problems = spark.createDataFrame(
    [("M1", -1, Decimal("-5.00"))],
    "line_id STRING, quantity INT, unit_price DECIMAL(12,2)")

two_problems.withColumn("reasons", F.array_compact(F.array(
    F.when(F.col("quantity") <= 0,  F.lit("bad_quantity")),
    F.when(F.col("unit_price") < 0, F.lit("bad_price")),
))).show(truncate=False)

### Task 7 · Write the outputs, report and check the totals

**Problem:**
1. Write the clean and quarantine lines to **separate Parquet** folders, keeping the original columns (plus `reasons` for quarantine).
2. Build a **rule-count report**: how many lines broke each rule? `explode` turns each array item into its own row, so the reasons can be counted.
3. Read both outputs back and check that **raw = clean + quarantine**.

**Expected:**
- Report: `bad_quantity` 3 · `bad_discount` 3 · `unknown_product` 2 · `missing_order_id` 2.
- **785 = 775 + 10** → `True`. The quarantine rate is about 1.27%.
- The 775 clean lines still include **8 duplicates**. Day 05 removes them, leaving 767 (the manifest's `clean_unique_lines`). Day 05 reads `clean_lines` from here.

In [0]:
# Clean lines: the original columns only
(accepted
    .select("line_id", "order_id", "product_id", "quantity", "unit_price", "discount_pct", "source_file")
    .write.mode("overwrite").parquet(f"{OUT}/clean_lines"))

# Quarantine lines: the original columns plus the reasons
(quarantine
    .select("line_id", "order_id", "product_id", "quantity", "unit_price", "discount_pct", "source_file", "reasons")
    .write.mode("overwrite").parquet(f"{OUT}/quarantine_lines"))

In [0]:
# explode: one row per reason, then count each reason
report = quarantine.select(F.explode("reasons").alias("reason")).groupBy("reason").count()
report.show()

report.write.mode("overwrite").option("header", True).csv(f"{OUT}/rule_counts")

In [0]:
# Read the outputs back and check that every raw line landed in exactly one place
clean_count      = spark.read.parquet(f"{OUT}/clean_lines").count()
quarantine_count = spark.read.parquet(f"{OUT}/quarantine_lines").count()
raw_count        = order_items.count()

print("raw        :", raw_count)
print("clean      :", clean_count)
print("quarantine :", quarantine_count)
print("raw == clean + quarantine ?", raw_count == clean_count + quarantine_count)
print("quarantine rate:", round(100 * quarantine_count / raw_count, 2), "%")

### Key takeaways
- **Quarantine, don't delete.** Bad lines are kept with their reasons, so they can be fixed and reprocessed. Nothing disappears silently.
- **Parse problem vs business problem:**
  - A parse problem means the text couldn't be read, e.g. `two` → cast failed (Task 3).
  - A business problem means the value was read fine but is wrong, e.g. quantity −1.
- **Clean text** with `trim` (spaces), `lower` / `initcap` (case). Never change IDs, and keep the original values next to the cleaned ones.
- **Missing = null OR blank.** Check both. `coalesce` picks the first non-null value.
- **Not every missing value is a rejection.** A missing email is fine; a missing ID is not.
- **The null trap:** `quantity <= 0` is null, not true, when quantity is null, so every rule needs an explicit `isNull()` check.
- **left_anti join** finds the "not found" rows (unknown products and orders). A null key never matches.
- **One array of reasons per line** keeps every problem. `explode` + `groupBy` turns it into a report.
- **Always reconcile:** raw = clean + quarantine. The clean rows still contain duplicates; that's Day 05.

*Skipped on purpose (they don't add a new concept): severity categories, failure-threshold policy, reprocessing workflow, and the reusable validator function.*